[Reference](https://medium.com/@pankaj_pandey/5647e6bfb4b1)

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict

class S(TypedDict):
    user_id: str
    user_msg: str
    user_ctx: Dict
    need_rag: bool
    evidence: List[Dict]
    answer: str

CAG_PACK = """# Support CAG Pack (cached)
- Role & scope, refusal policy
- Refund & warranty matrix; SLAs; escalation tree
- Tools: get_user_profile(uid), list_orders(uid), create_rma(order_id)
- Output schema: ResolutionPlan { intent, steps[], create_ticket, citation_ids[] }
- Few-shots: (6 good; 2 counter-examples)
"""

def load_user_ctx(s: S) -> S:
    s["user_ctx"] = {
        "profile": get_user_profile(s["user_id"]),
        "orders": list_orders(s["user_id"]),
        "open_tickets": list_open_tickets(s["user_id"]),
    }
    return s

def gate(s: S) -> S:
    q = s["user_msg"].lower()
    s["need_rag"] = any(t in q for t in ["latest","updated","policy","terms","2025","outage","as of","version"])
    return s

def retrieve(s: S) -> S:
    if not s["need_rag"]: return s
    s["evidence"] = contextual_retrieve(s["user_msg"], k=8)  # dense+BM25+rerank
    return s

def synthesize(s: S) -> S:
    msgs = [("system", CAG_PACK),
            ("system", f"USER_CONTEXT:\n{shorten(s['user_ctx'])}"),
            ("user", s["user_msg"])]
    if s.get("evidence"): msgs.append(("system", format_citations(s["evidence"])))
    s["answer"] = llm_chat(msgs, response_schema="ResolutionPlan")  # Structured outputs
    return s

g = StateGraph(S)
g.add_node("load_user_ctx", load_user_ctx)
g.add_node("gate", gate)
g.add_node("retrieve", retrieve)
g.add_node("synthesize", synthesize)
g.add_edge(START, "load_user_ctx"); g.add_edge("load_user_ctx", "gate")
g.add_conditional_edges("gate", lambda s: "retrieve" if s["need_rag"] else "synthesize",
                        {"retrieve": "retrieve", "synthesize": "synthesize"})
g.add_edge("retrieve", "synthesize"); g.add_edge("synthesize", END)
app = g.compile()